# Setup — load model, tokenizer, and a pushT batch

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch 
from dreamerv4uwm.datasets import ShardedHDF5Dataset
from dreamerv4uwm.models.utils import load_tokenizer
from dreamerv4uwm.models.utils import load_denoiser
# DATA_PATH = "/home/mim-server/datasets/soar_data_sharded"
# DATA_PATH = "/home/mim-server/datasets/Finger/H5/combined"
DATA_PATH = "/scratch/rk4342/datasets/benchmarks/ogbench/manipulation/visual-puzzle-4x4-v0"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resolution = (256, 256)

In [3]:
from hydra import initialize, compose
from omegaconf import OmegaConf
with initialize(version_base=None, config_path="scripts/config"):
    cfg = compose(config_name="dynamics/ogbench-manipulation.yaml")

/ext3/miniconda/envs/dreamerv4/lib/python3.11/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'dynamics/ogbench-manipulation.yaml': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)


In [ ]:
cfg.denoiser.train_reward_model = True

In [ ]:
is_2x_temporal = True

if is_2x_temporal:
    cfg.denoiser.layer_types = ["spatial", "temporal", "spatial", "temporal"]
    dynamics_ckpt = "/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/dynamics/pushT/no-shortcuts/wm-policy-2x-temporal-293044.pt"
else:
    cfg.denoiser.layer_types = ["spatial", "spatial", "spatial", "temporal"]
    dynamics_ckpt = "/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/dynamics/pushT/no-shortcuts/wm-policy-1x-temporal-293044.pt"


In [ ]:
# dynamics_ckpt = "/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/dynamics/pushT/no-shortcuts/all-2x-temporal-485204.pt"
dynamics_ckpt = "/scratch/rk4342/projects/dreamerV4-UWM/checkpoints/dynamics/pushT-reward-demo-play-mix/51060.pt"

In [4]:
tokenizer_ckpt="/scratch/rk4342/projects/dreamerV4-UWM/checkpoints/tokenizer/ogbench-manipulation/23000.pt"
# cfg.dynamics_ckpt = dynamics_ckpt
cfg.tokenizer_ckpt=tokenizer_ckpt
# denoiser = load_denoiser(cfg, device, max_num_forward_steps=300)
tokenizer = load_tokenizer(cfg, device, max_num_forward_steps=300)
tokenizer = tokenizer.eval().cuda()
# denoiser = denoiser.eval().cuda()

In [5]:
import mediapy
from torch.nn.functional import interpolate

dataset = ShardedHDF5Dataset(
        data_dir=DATA_PATH,
        window_size=64,
        stride=1,
        split='train',
        train_fraction=0.9,
        split_seed=123,
    )

# Arbitrary index into that episode
batch = dataset[torch.randint(len(dataset), (1,)).item()]
# imgs = batch["image"][:,[2, 1, 0], :, :]  # (T, C, H, W)
imgs = batch["image"]  # (T, C, H, W)
actions = batch["action"][:,:cfg.denoiser.n_actions]  # (1, T, N_lat, D_lat)
# actions=torch.zeros_like(actions)
imgs = interpolate(imgs, resolution).to(device=device)[None] # resize to tokenizer resolution

with torch.no_grad():
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        latents = tokenizer.encode(imgs)
        imgs_recon = tokenizer.decode(latents)

from mediapy import show_video
def plotVideo(video):
    imgs = video.cpu().permute(0,2,3,1).to(torch.float32).numpy()*255
    imgs = imgs.astype('uint8')
    mediapy.show_video(imgs, fps=10)

plotVideo(imgs_recon[0])
plotVideo(imgs[0])

Train split: 213944 windows from 1138 episodes


This browser does not support the video tag.


This browser does not support the video tag.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plotSnapshots(video, n_frames=5, border_width=2, figsize=None):
    """Plot n_frames evenly-spaced snapshots from a video tensor, separated by black borders.

    Args:
        video: (T, C, H, W) tensor with values in [0, 1].
        n_frames: number of frames to sample.
        border_width: width of black separator lines in pixels.
    """
    T = video.shape[0]
    indices = np.linspace(0, T - 1, n_frames, dtype=int)
    frames = video[indices].cpu().permute(0, 2, 3, 1).to(torch.float32).numpy()
    frames = np.clip(frames * 255, 0, 255).astype(np.uint8)
    H, W, C = frames.shape[1], frames.shape[2], frames.shape[3]
    border = np.zeros((H, border_width, C), dtype=np.uint8)
    parts = []
    for i, f in enumerate(frames):
        if i > 0:
            parts.append(border)
        parts.append(f)
    strip = np.concatenate(parts, axis=1)
    if figsize is None:
        figsize = (n_frames * 3, 3)
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(strip)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


def plotActions(actions, figsize=None):
    act = actions.cpu().float().numpy() if hasattr(actions, 'cpu') else np.array(actions)
    figsize = (3, 3)
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.plot(act[:, 0], linewidth=3, label=r'$\mathbf{v_x}$')
    ax.plot(act[:, 1], linewidth=3, label=r'$\mathbf{v_y}$')
    ax.set_ylim(-1., 1.)
    ax.set_xticklabels([])
    ax.tick_params(axis='x', length=0)
    ax.tick_params(axis='y', labelsize=16)
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    ax.legend(fontsize=16)
    plt.tight_layout()
    plt.show()


# Autoregressive Sampling — `AutoRegressiveForwardDynamics`

KV-cached, frame-by-frame rollout for the **flow-matching** UWM denoiser. The class wraps the denoiser and tokenizer together with their decoder caches and supports two modes:

- **`wm`** — world model. The caller passes the next action at every `step()`; the action is treated as clean and only the next observation is denoised.
- **`policy`** — `step()` takes no input; the next observation and the next action are denoised jointly. Context actions are slightly noised at `tau_cond` to match policy training.

Both modes prime the cache via `reset(imgs_init, actions_init)` and then advance one frame per `step()`.


In [ ]:
from dreamerv4uwm.sampling import AutoRegressiveForwardDynamics
episode_length = 128
demo_end_index = 3
num_pred_steps = episode_length - demo_end_index
T_ctx = demo_end_index

denoising_step_count = 16        # power of two; flow-matching Euler steps per frame
context_cond_tau = 0.99
# max_forward_steps = T_ctx + num_pred_steps + 4   # safety margin for KV cache size
max_forward_steps = 64   # safety margin for KV cache size

imgs_ctx = imgs[:, :T_ctx]                      # (1, T_ctx, C, H, W)
actions_ctx = actions[:T_ctx][None].to(device)  # (1, T_ctx, n_act)

N_batch = 4
imgs_ctx = imgs_ctx.expand(N_batch, -1, -1, -1, -1)
actions_ctx = actions_ctx.expand(N_batch, -1, -1)

## 1. World-model rollout (`mode='wm'`)
We supply the next action at every step — actions are treated as clean and only the next obs is denoised. We feed the ground-truth future actions for the visualisation.


In [ ]:
ar_wm = AutoRegressiveForwardDynamics(
    denoiser=denoiser,
    tokenizer=tokenizer,
    mode='wm',
    context_length=max_forward_steps,
    max_forward_steps=max_forward_steps,
    context_cond_tau=context_cond_tau,
    denoising_step_count=denoising_step_count,
    device=device,
    dtype=torch.float32,
)

future_actions_wm = torch.zeros(1, num_pred_steps, actions_ctx.shape[-1], device=device)  # (1, T_pred, n_act)
future_actions_wm = future_actions_wm * 0.0   # zero-action rollout for visualisation
future_actions_wm[:, :, 0] = -0.00
future_actions_wm[:, :, 1] = 0.0
frames_wm = []
with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    ar_wm.reset(imgs_ctx, actions_ctx)
    for t in range(num_pred_steps):
        img_t = ar_wm.step(actions_t=future_actions_wm[:, t])
        frames_wm.append(img_t.detach().cpu())

video_wm = torch.stack(frames_wm, dim=1)[0]      # (num_pred_steps, C, H, W)
plotVideo(video_wm.to(torch.float32))
plotSnapshots(video_wm.to(torch.float32), n_frames=5)
plotActions(future_actions_wm[0])


## 2. Policy rollout (`mode='policy'`)
No actions are fed during rollout — the policy denoises the next obs and the next action jointly at each step.


In [ ]:
ar_pol = AutoRegressiveForwardDynamics(
    denoiser=denoiser,
    tokenizer=tokenizer,
    mode='policy',
    context_length=max_forward_steps,
    max_forward_steps=max_forward_steps,
    context_cond_tau=context_cond_tau,
    denoising_step_count=denoising_step_count,
    device=device,
    dtype=torch.float32,
)



In [ ]:
frames_pol, acts_pol = [], []
with torch.no_grad():
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        ar_pol.reset(imgs_ctx, actions_ctx)
        for t in range(num_pred_steps):
            img_t, act_t = ar_pol.step()
            frames_pol.append(img_t.detach().cpu())
            acts_pol.append(act_t.detach().cpu())

In [ ]:
video_pol = torch.stack(frames_pol, dim=1)
acts_pol_seq = torch.stack(acts_pol, dim=1)

video_all = torch.concat([imgs[:, :T_ctx].cpu().expand(video_pol.shape[0], -1, -1, -1, -1), video_pol.cpu()], dim=1)  # (1, 3, T, C, H, W)
# actions_all = torch.concat([actions[:T_ctx].cpu(), acts_pol_seq.cpu()], dim=0)  # (1, T, n_act)
for k in range(video_all.shape[0]):
    plotVideo(video_all.to(torch.float32)[k])

# plotSnapshots(video_all.to(torch.float32), n_frames=5)
# plotActions(acts_pol_seq.to(torch.float32))